# Mnemonics LongMemEval — Stage-2 (session-level doc filter)

Hiçbir şey yüklemenize gerek yok. Dataset HuggingFace'ten otomatik inecek (~265 MB).  
**Runtime → Change runtime type → T4 GPU** seçin, sonra tüm hücreleri sırayla çalıştırın.

**Ablation:** aynı sorular iki kez — biri `--use-doc-filter` KAPALI (mevcut baseline: R@1=0.846 / R@10=0.898 hedef), biri AÇIK (Stage-2 hedefi: R@1>0.88 / R@10>0.92 50q; R@1>0.90 / R@10>0.95 500q).

## 1) Repo + bağımlılıklar

In [ ]:
!git clone https://github.com/nakata-app/mnemonics.git /content/mnemonics
%cd /content/mnemonics
!git log --oneline -5

In [ ]:
!pip install -q -e . sentence-transformers numpy adaptmem 2>&1 | tail -5
print('Install done')

## 2) Dataset indir (HuggingFace → local JSON)

In [ ]:
import os
DATA = '/content/longmemeval_s_cleaned.json'
if not os.path.exists(DATA) or os.path.getsize(DATA) < 1_000_000:
    print('Downloading dataset (~277 MB)...')
    !rm -f {DATA}
    !wget -q --show-progress -O {DATA} \
        'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json'
else:
    print('Dataset zaten var, atlanıyor')
print(f'Size: {os.path.getsize(DATA)/1e6:.1f} MB')

In [ ]:
# Eval script DATA path'ini patch'le
import re, pathlib
p = pathlib.Path('/content/mnemonics/benchmarks/longmemeval_eval.py')
src = p.read_text()
src = re.sub(r'DATA = Path\([^)]+\)', f'DATA = Path("{DATA}")', src)
p.write_text(src)
!grep -n 'DATA = Path' {p}
print('Patch OK')

## 3) Smoke test — 5 soru, hızlı kontrol (doc_filter ON)

In [ ]:
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 5 --mode no_rerank --use-doc-filter \
    --augment-preferences --candidate-k 50 \
    --out /tmp/smoke.json && echo '=== SMOKE OK ==='

## 4) 50q ablasyon — baseline (doc_filter OFF) vs Stage-2 (doc_filter ON)

Aynı 50 soruyu iki kez, aynı seed. Delta = session-level filtrenin net katkısı.

In [ ]:
os.makedirs('/content/results', exist_ok=True)

# 4A) Baseline: doc_filter OFF
print('=== 4A: 50q baseline (doc_filter OFF) ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme50_baseline.json \
    --per-q-out /content/results/lme50_baseline_perq.json

In [ ]:
# 4B) Stage-2: doc_filter ON
print('=== 4B: 50q Stage-2 (doc_filter ON) ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank --use-doc-filter \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme50_stage2.json \
    --per-q-out /content/results/lme50_stage2_perq.json

In [ ]:
# 4C) Delta tablosu
import json
b = json.load(open('/content/results/lme50_baseline.json'))['mnemonics_rerank']
s = json.load(open('/content/results/lme50_stage2.json'))['mnemonics_rerank']
print(f'{"Metrik":8} {"Baseline":>10} {"Stage-2":>10} {"Delta":>8}')
print('-' * 40)
for k in ('R@1', 'R@5', 'R@10'):
    delta = s[k] - b[k]
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'{k:8} {b[k]:>10.3f} {s[k]:>10.3f} {delta:>+7.3f} {arrow}')
print()
print('HEDEF: R@1 >= 0.88 ve R@10 >= 0.92')
ok = s['R@1'] >= 0.88 and s['R@10'] >= 0.92
print('SONUÇ:', 'PASS ✅' if ok else 'FAIL ❌')

In [ ]:
# 4D) Kurtarılan vs kaybedilen sorular (R@10)
pb = {r['qid']: r for r in json.load(open('/content/results/lme50_baseline_perq.json'))}
ps = {r['qid']: r for r in json.load(open('/content/results/lme50_stage2_perq.json'))}
rescued, broken = [], []
for qid in pb:
    if qid not in ps: continue
    if ps[qid]['hit@10'] and not pb[qid]['hit@10']:
        rescued.append((qid, pb[qid]['qtype'], pb[qid]['question'][:90]))
    elif pb[qid]['hit@10'] and not ps[qid]['hit@10']:
        broken.append((qid, pb[qid]['qtype'], pb[qid]['question'][:90]))

print(f'RESCUED — Stage-2 ile yakalandı ({len(rescued)} soru):')
for qid, t, q in rescued:
    print(f'  [{t}] {q}')
print(f'\nBROKEN — Stage-2 ile kaybedildi ({len(broken)} soru):')
for qid, t, q in broken:
    print(f'  [{t}] {q}')

## 5) 500q full eval — baseline + Stage-2

T4 GPU'da ~30-45 dk her biri. İkisi sıralı, toplamda ~1-1.5 saat. Hedef: R@1>0.90 / R@10>0.95.

In [ ]:
# 5A) 500q baseline
print('=== 5A: 500q baseline (doc_filter OFF) ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme500_baseline.json \
    --per-q-out /content/results/lme500_baseline_perq.json

In [ ]:
# 5B) 500q Stage-2 doc_filter ON
print('=== 5B: 500q Stage-2 (doc_filter ON) ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank --use-doc-filter \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme500_stage2.json \
    --per-q-out /content/results/lme500_stage2_perq.json

In [ ]:
# 5C) 500q delta + type breakdown
b5 = json.load(open('/content/results/lme500_baseline.json'))['mnemonics_rerank']
s5 = json.load(open('/content/results/lme500_stage2.json'))['mnemonics_rerank']
print('=== OVERALL ===')
for k in ('R@1', 'R@5', 'R@10'):
    print(f'{k:6}  baseline={b5[k]:.3f}  stage2={s5[k]:.3f}  Δ={s5[k]-b5[k]:+.3f}')
print()
print('HEDEF: R@1 > 0.90 ve R@10 > 0.95')
ok = s5['R@1'] > 0.90 and s5['R@10'] > 0.95
print('SONUÇ:', 'PASS ✅' if ok else 'FAIL ❌')
print()
print('MemPalace ref: R@1=0.920 / R@5=0.960 / R@10=1.000')
print()
print('=== BY TYPE (R@10) ===')
for qt in sorted(b5['by_type']):
    bb = b5['by_type'][qt]
    ss = s5['by_type'].get(qt, {})
    d = ss.get('R@10', 0) - bb['R@10']
    print(f'{qt:25} n={bb["n"]:3}  base={bb["R@10"]:.3f}  stage2={ss.get("R@10",0):.3f}  Δ={d:+.3f}')

In [ ]:
# 5D) Sonuçları Drive'a yedekle (Drive mount açıksa)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil, os
    dest = '/content/drive/MyDrive/lme/results'
    os.makedirs(dest, exist_ok=True)
    for f in os.listdir('/content/results'):
        shutil.copy(f'/content/results/{f}', f'{dest}/{f}')
    print('Drive backup OK:', dest)
except Exception as e:
    print('Drive yok veya mount edilmedi, sonuçlar /content/results/ de:')
    !ls -lh /content/results/